# Walkthrough: MedEinst / ECR-Agent (arxiv 2601.06636)

Paper: Chen, Zhu, Huang, Wang — *MedEinst: Benchmarking the Einstellung Effect in Medical LLMs through Counterfactual Differential Diagnosis*.

Maps §4 / Algorithm 2 / §3.5 onto `src/` with CPU-only checks. `demo_llm()` needs no API key. MedEinst pairs are unreleased; MCR400 comes from the parent repo.

Read `REPRODUCTION_NOTES.md` before treating numbers as a Table 1 reproduction.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("ROOT", ROOT)


## Dual-Pathway Perception (§4.2.1)

> "we decouple statistical priors from factual observation through two parallel pathways. Firstly, the *intuitive pathway* generates Top-*k* candidate diagnoses *Dset* ... Secondly, the *analytic pathway* produces a *problem representation* ... status *s(p)* as *Present*, *Absent*, or *Missing*."

Appendix C: **k = 5**. Table A7: Absent only if the text says no / denies / without.


In [ ]:
from src.llm import demo_llm
from src.model import DualPathwayPerception, ModelConfig

cfg = ModelConfig(top_k=5, live_search_pubmed=False, live_search_opentargets=False)
perc = DualPathwayPerception(demo_llm(), top_k=cfg.top_k)
x = "22M sudden knife-like chest pain and dyspnea. I have had a DVT. I have no fever."
dset, _ = perc.intuitive(x)
one_liner, p_obs = perc.analytic(x)
assert len(dset) == 5, dset
assert any(p.status == "Absent" for p in p_obs), p_obs
print("Dset", dset)
print("Pobs", [(p.content, p.status) for p in p_obs])


## Merge-or-prune (Eq. 2, Appendix A.3)

$$\mathrm{Action}(p_{\mathrm{script}})=\begin{cases}\mathrm{Merge} & \cos(e_{p\mathrm{script}}, e_{p\mathrm{obs}}) > \tau \\\mathrm{Prune} & \text{otherwise}\end{cases}$$

with **τ = 0.9**. Embedding model is [UNSPECIFIED]; we use bag-of-words cosine.


In [ ]:
from src.model import GraphNode, PNode, merge_or_prune
from src.utils import pairwise_cosine

tau = 0.9
same = pairwise_cosine("prior DVT", "prior DVT")
assert same > tau, same
script = [GraphNode(id="s1", kind="patient", content="prior DVT")]
obs = [PNode(id="p2", content="history of DVT", original_text="DVT", status="Present")]
kept = merge_or_prune(script, obs, tau)
assert kept, kept
print("cos identical", same, "kept", [n.content for n in kept])


## Evidence score S(d) (§4.2.3)

$$S(d) = w_m N_{\mathrm{match}}(d) - w_c N_{\mathrm{conf}}(d) - w_s N_{\mathrm{shadow}}(d)$$

Weights **w_m, w_c, w_s** are [UNSPECIFIED]; config defaults are 1.0.


In [ ]:
from src.loss import evidence_score
from src.model import CausalGraph, GraphEdge, GraphNode

g = CausalGraph(disease="pulmonary embolism")
g.add_node(GraphNode(id="d_pe", kind="disease", content="pulmonary embolism"))
g.add_node(GraphNode(id="k1", kind="knowledge", content="prior DVT", ktype="Pivot"))
g.add_node(GraphNode(id="sh", kind="shadow", content="family pneumothorax"))
g.add_edge(GraphEdge("p2", "k1", "matching"))
g.add_edge(GraphEdge("sh", "d_pe", "penalty"))
s = evidence_score(g, "pulmonary embolism", 1.0, 1.0, 1.0)
assert s == 0.0, s  # 1 match - 0 conf - 1 shadow
print("S(PE) =", s)


## Metrics (§3.5)

- Acc_base = |S_correct_control| / N_total
- Acc_rob = fraction of pairs with f(x^c)=ygt and f(x^t)=ybias
- R_bias = among correct controls, fraction with f(x^t)=ygt (still the *control* label)

Definition 1's equation was blank in the PDF extract; R_bias operationalizes the trap as predicting ygt on x^t.


In [ ]:
from src.data import Case
from src.evaluate import evaluate_cases, baseline_accuracy, bias_trap_rate, robust_accuracy

assert abs(baseline_accuracy(2, 4) - 0.5) < 1e-12
assert abs(robust_accuracy(1, 4) - 0.25) < 1e-12
assert abs(bias_trap_rate(1, 2) - 0.5) < 1e-12

pairs = [
    Case("1", "c", "pneumothorax", "toy", x_c="c", x_t="t", y_bias="PE"),
    Case("2", "c2", "pneumothorax", "toy", x_c="c2", x_t="t2", y_bias="PE"),
]

def predict(text):
    return {"c": "pneumothorax", "t": "pneumothorax", "c2": "pneumothorax", "t2": "PE"}[text]

rep = evaluate_cases(pairs, predict)
assert not rep.unpaired
assert abs(rep.acc_base - 1.0) < 1e-12
assert abs(rep.acc_rob - 0.5) < 1e-12
assert abs(rep.r_bias - 0.5) < 1e-12
print(rep)


## DCI pipeline (Algorithm 2 lines 17–50)

IntuitivePathway → AnalyticPathway → for each d: init / LiveSearch / backward shadows → CalculateScore → LLM_Judge.


In [ ]:
from src.model import ECRAgent

agent = ECRAgent(llm=demo_llm(), config=cfg)
result = agent.dci_pipeline(x)
assert result.diagnosis.lower() == "pulmonary embolism", result.diagnosis
assert len(result.dset) == 5
print("diagnosis", result.diagnosis)
print("scores", result.scores)


## MCR400 substitute for MedEinst (§3.3 vs user instruction)

Paper test set: 5,383 pairs. Here: `mcr_val_seq100_v1` + `mcr_val_seq100_v2` + `mcr_val_seq200b_v1` = 400 unpaired cases. Acc_rob and R_bias are `None`.


In [ ]:
from src.data import MCR400Dataset
from src.evaluate import evaluate_cases

parent = ROOT.parent
ds = MCR400Dataset(parent)
assert len(ds) == 400, len(ds)
assert all(not c.is_pair for c in ds.cases)
rep = evaluate_cases(ds.cases[:3], predict=lambda _x: ds.cases[0].y_gt)
assert rep.unpaired and rep.acc_rob is None and rep.r_bias is None
print("n", len(ds), "slice0", ds[0].slice_name, "gold0", ds[0].y_gt)
print(rep)
